# small_fable — CONTRASTIVE (plan-conditioned) SFT

Fixes the v3 failure (7 distinct plans / 1000 rows, `I(plan;answer)≈0`, negative ablation gap).
Here the **plan is an executable program** (`SET 28 MUL 4 MUL 31`) and `answer == execute(plan)`, so
plan↔answer mutual information is maximal by construction. Each row ships **3 oracle-verified hard
negatives** (one step perturbed, answer genuinely changed).

Loss per example over {gold plan, 3 hard-neg plans}:

$$L = \underbrace{\mathrm{CE}(a^*\mid \text{instr},\text{plan})}_{\text{(1) fit answer}} + \beta\underbrace{\mathrm{InfoNCE}}_{\text{(2) MI}} + \lambda_{kl}\underbrace{\mathrm{KL}(\text{exec}\,\|\,\text{base})}_{\text{(3) anchor}}$$

where InfoNCE $= -\log \frac{e^{s_{gold}}}{e^{s_{gold}}+\sum_k e^{s_{neg_k}}}$, $s_x=-\mathrm{CE}(a^*\mid\text{instr},\text{plan}_x)/\tau$ — a lower bound on $I(\text{plan};\text{answer}\mid\text{instr})$.

**Watch the held line:** `ablation_gap` (with-plan minus no-plan accuracy) should go **positive** — the executor genuinely needs the plan.

## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — Runtime → Change runtime type → T4 GPU, then re-run.'

## 1 · Clone & install

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt
!pip uninstall -y torchao >/dev/null 2>&1; echo 'torchao removed (peft clash workaround)'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('\nReady.')

## 2 · Build the v4 executable-plan corpus (fast, CPU)
1000 examples × 3 verified hard negatives. Re-run the generator so the dataset is always fresh.

In [ ]:
!python -u tools/gen_synth_v4.py --n 1000 --k 3 --out dataset/sft_synth_v4.jsonl \
    --val_n 25 --val_out dataset/sft_synth_v4_val.jsonl
import json, collections
rows = [json.loads(l) for l in open('dataset/sft_synth_v4.jsonl')]
plans = {r['plan_str'] for r in rows}
flip = sum(hn['answer'] != r['answer'] for r in rows for hn in r['hard_negatives'])
tot  = sum(len(r['hard_negatives']) for r in rows)
print(f'rows={len(rows)}  distinct plans={len(plans)}  (v3 was 7)  param-edit answer-flip={flip/tot:.0%}')
print('sample:', rows[0]['plan_str'], '->', rows[0]['answer'],
      '| neg:', rows[0]['hard_negatives'][0]['plan_str'], '->', rows[0]['hard_negatives'][0]['answer'])

## 3 · Contrastive SFT  (CE + InfoNCE/MI + KL)
On A100 use `--dtype bf16`; on a T4 use `--dtype fp16`. `--beta` MI weight, `--lam_kl` base-anchor,
`--tau` InfoNCE temperature. `--kl_ref disable_adapter` (default) takes the KL reference from the
original Qwen via adapter-off (0 extra memory; numerically identical on Qwen2.5). `--kl_ref frozen`
loads a separate untouched Qwen as the reference (unambiguous, +~3GB; use on A100).

**The held line is the verdict.** It prints `acc_plan / neg_follow / neg_to_gold / acc_noplan / ablation_gap`:
- **`neg_follow` HIGH** = given a *wrong* plan, the model emits *that wrong plan's* answer -> it is faithfully executing the plan.
- **`neg_to_gold` LOW** = it does *not* fall back to the instruction's gold answer when given a wrong plan.
- **Planning is causally load-bearing iff `neg_follow` HIGH and `neg_to_gold` LOW** (e.g. neg_follow >= 0.6 and neg_follow - neg_to_gold >= 0.3).
  `acc_plan`/`ablation_gap` are *supporting* only: the answer is derivable from the instruction, so they can look good even if the plan is ignored.
- Also watch **`gap(CEneg-CEpos)`** scroll positive during training -- the MI term biting.

In [ ]:
DTYPE = 'fp16'  # set to 'bf16' on A100
!python -u train_sft_contrastive.py \
    --data dataset/sft_synth_v4.jsonl \
    --train 900 --held 100 --eval_n 40 \
    --epochs 3 --bs 2 --lr 2e-5 \
    --k 3 --beta 1.0 --tau 1.0 --lam_kl 0.1 \
    --kl_ref disable_adapter --val dataset/sft_synth_v4_val.jsonl \
    --dtype $DTYPE --device cuda --out sft_contrastive_ckpt


## 4 · Inspect: is the plan load-bearing AND faithful?
Decode the same problem three ways: with the **gold** plan, with **no** plan, and with a **hard-negative**
plan. A faithful executor: gold→gold answer, neg→the *negative's* answer (it follows the wrong plan),
no-plan→worse. That neg-following is exactly what the InfoNCE term trained.

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import train_sft_contrastive as M

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained('sft_contrastive_ckpt')
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16).cuda()
model = PeftModel.from_pretrained(base, 'sft_contrastive_ckpt').cuda().eval()

def gen(prompt):
    ids = tok(prompt, add_special_tokens=False, return_tensors='pt')['input_ids'].cuda()
    out = model.generate(ids, max_new_tokens=16, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, ids.size(1):], skip_special_tokens=True).strip()

rows = [json.loads(l) for l in open('dataset/sft_synth_v4.jsonl')]
for r in rows[-5:]:
    hn = r['hard_negatives'][0]
    print('Q:', r['instruction'][:80])
    print(f"   gold plan {r['plan_str']:<28} -> {gen(M.prompt_with_plan(r['instruction'], r['plan_str']))!r:>10}  (gold={r['answer']})")
    print(f"   neg  plan {hn['plan_str']:<28} -> {gen(M.prompt_with_plan(r['instruction'], hn['plan_str']))!r:>10}  (neg ={hn['answer']})")
    print(f"   no  plan {'':<28} -> {gen(M.prompt_no_plan(r['instruction']))!r:>10}")
    print()

## 5 · Train-vs-validation error curves
`train_sft_contrastive.py` saved `curves.png` + `history.json` into the checkpoint dir. Panel **(A)**
is the classic error plot — train CE vs the 25-row holdout val CE (gap opening = overfitting). Panel
**(B)** is the verdict: `neg_follow` rising and `neg_to_gold` falling across epochs = the plan is
becoming causally load-bearing.

In [ ]:
import os, json
from IPython.display import Image, display
CKPT = 'sft_contrastive_ckpt'
png, hist = os.path.join(CKPT,'curves.png'), os.path.join(CKPT,'history.json')
if os.path.exists(png):
    display(Image(filename=png))
else:
    print(f'[!] {png} not found - did training finish and call plot_curves(...)?')
if os.path.exists(hist):
    ve = json.load(open(hist)).get('val_epochs', [])
    if ve:
        print('final validation metrics:'); print(json.dumps(ve[-1], indent=2))
        f = ve[-1]
        verdict = 'PLAN IS CAUSAL' if (f['neg_follow']>=0.6 and f['neg_follow']-f['neg_to_gold']>=0.3) else 'plan NOT yet load-bearing'
        print(f"\nVERDICT: {verdict}  (neg_follow={f['neg_follow']:.2f}, neg_to_gold={f['neg_to_gold']:.2f})")
